# FoodChain AI - Feature Engineering

## Objective

Transform the raw supplier dataset into ML-ready features for the supplier recommendation engine.

### Workflow

1. Load dataset
2. Simulate vendor location
3. Calculate Haversine distance
4. Filter suppliers within delivery radius
5. Select ML features
6. Scale features
7. Prepare feature matrix
8. Save scaler
9. Key observations


In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/pune_supplier_dataset.csv")
df.head()


## Dataset Shape

In [ ]:
print("Shape:", df.shape)
print(df.columns.tolist())

## Simulate Vendor Location

Later this latitude and longitude will come from the frontend after the vendor grants GPS permission.


In [ ]:
vendor_lat = 18.5204
vendor_lon = 73.8567

## Haversine Distance Function

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2-lat1)
    dlon = radians(lon2-lon1)

    a = (sin(dlat/2)**2 +
         cos(radians(lat1))*cos(radians(lat2))*
         sin(dlon/2)**2)

    c = 2*atan2(sqrt(a), sqrt(1-a))
    return R*c

df["distance_km"] = df.apply(
    lambda x: haversine_distance(
        vendor_lat,
        vendor_lon,
        x["latitude"],
        x["longitude"]
    ),
    axis=1
)

df[["supplier_name","ingredient","distance_km"]].head()


## Filter Suppliers Within Delivery Radius

In [ ]:
filtered_df = df[
    df["distance_km"] <= df["delivery_radius_km"]
].copy()

print("Original Rows :", len(df))
print("Filtered Rows :", len(filtered_df))


## Feature Selection

In [ ]:
features = [
    "price",
    "distance_km",
    "rating",
    "quality_score",
    "reliability_score",
    "average_delivery_time_min"
]

X = filtered_df[features].copy()
X.head()


## Feature Scaling

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=features
)

X_scaled.head()


## Verify Scaling

In [ ]:
X_scaled.describe().T

## Save Scaler

In [ ]:
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

print("Scaler saved successfully.")


## Reusable Feature Engineering Function

In [ ]:
def prepare_features(df, vendor_lat, vendor_lon):
    data = df.copy()

    data["distance_km"] = data.apply(
        lambda x: haversine_distance(
            vendor_lat,
            vendor_lon,
            x["latitude"],
            x["longitude"]
        ),
        axis=1
    )

    data = data[
        data["distance_km"] <= data["delivery_radius_km"]
    ].copy()

    X = data[[
        "price",
        "distance_km",
        "rating",
        "quality_score",
        "reliability_score",
        "average_delivery_time_min"
    ]]

    X = scaler.transform(X)

    return data, X


# Key Observations

- Haversine distance successfully converts GPS coordinates into a meaningful ML feature.
- Suppliers outside their delivery radius are removed before clustering.
- Six numerical features are selected for K-Means.
- Features are standardized using StandardScaler.
- The preprocessing pipeline is reusable and can directly receive vendor GPS coordinates from the frontend or FastAPI.
- The processed feature matrix is now ready for K-Means clustering.
